# 2026-10-21_6 MG 堆叠小提琴图（转置测试版）——ABCB11 / TTC39B / TFRC

**需求与口径**（相对 `2026-10-21_5_..._sharedY.pdf` stage 版**旋转 90°**，测试版本，**不覆盖任何现有输出**）

- 布局转置：**X 轴（顶部条带）= 基因**（ABCB11、TTC39B、TFRC，左->右）；**Y 轴 = 分组**（Early-central 自上而下 → Late-peripheral；成年页 young-central → old-peripheral）；小提琴横向（表达值沿 X，0 在左）
- 其余口径与 _5 stage 版一致：仅 MG；**剔除 70years 与 30years**；胚胎/成年分页；配色 stage 本色 = central、浅色 = peripheral；sharedY（同一基因两页表达轴范围一致，上限跨两期全局计算）
- 实现：`VlnPlot(stack = TRUE, flip = FALSE)` + `geom_blank`（撑住每基因表达上限）+ 替换 X 标度（`limits = c(0, NA), oob = squish`）+ `scale_y_discrete(limits = rev(...))` 使 Early-central 位于顶部；基因条带文字水平显示（Seurat 默认 -90° 旋转，此处改为 0° 便于阅读）
- 仅输出 PDF，不输出 PNG

**数据源（复用每细胞表，不重读大对象）**

- `Data/2026-10-21_3_MG_gene_percell_expr_with70y.rds`（载入后剔除 70/30years）

**输出**

- `Plot/2026-10-21_6_MG_ABCB11_TTC39B_TFRC_stacked_violin_sharedY_rotated.pdf`（2 页：胚胎 / 成年，7×5.5 in）
- 原 `2026-10-21_5_..._sharedY.pdf` 等现有文件均不改动

In [ ]:
# ---- 1. 环境与参数 ----------------------------------------------------------
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
  library(scales)
})
set.seed(1)

DATE_STR <- "2026-10-21_6"
PROJ_DIR <- "/project/work/2026-10-21_violinplot"
DATA_DIR <- file.path(PROJ_DIR, "Data")
PLOT_DIR <- file.path(PROJ_DIR, "Plot")

PERCELL_RDS <- file.path(DATA_DIR, "2026-10-21_3_MG_gene_percell_expr_with70y.rds")
stopifnot(file.exists(PERCELL_RDS))

GENES <- c("ABCB11", "TTC39B", "TFRC")

# 剔除供体（与 _5 一致）：70years 仅 central、30years 仅 peripheral
EXCLUDE_AGES <- c("70years", "30years")

GROUP_LEVELS_EMB <- c("Early-central", "Early-peripheral", "Mid-central",
                      "Mid-peripheral", "Late-central", "Late-peripheral")
GROUP_LEVELS_ADU <- c("young-central", "young-peripheral", "old-central", "old-peripheral")

# ---- 配色（沿用 panelFH 的 stage/region 逻辑；用户提供）-----------------------
cols.stage.full <- c('#59A14F','#EDC948','#B07AA1','#4E79A7','#E15759')  ## 用户指定统一配色
names(cols.stage.full) <- c('Early','Mid','Late','Young','Old')
cols.stage      <- cols.stage.full[c("Early","Mid","Late")]                       ## dev: Early/Mid/Late
cols.adultstage <- setNames(cols.stage.full[c("Young","Old")], c("young","old"))  ## adult: 数据里为小写 young/old
lighten_col <- function(col, f = 0.45) {
  m <- grDevices::col2rgb(col)
  grDevices::rgb(t(m + (255 - m) * f), maxColorValue = 255)
}
## central = 本色, peripheral = 浅色
region_cols <- function(base) {
  v <- c(base, sapply(base, lighten_col))
  names(v) <- c(paste0(names(base), "_central"), paste0(names(base), "_peripheral"))
  v
}

cols.emb.groups <- region_cols(cols.stage)
names(cols.emb.groups) <- gsub("_", "-", names(cols.emb.groups))
cols.adu.groups <- region_cols(cols.adultstage)
names(cols.adu.groups) <- gsub("_", "-", names(cols.adu.groups))

cat("本次基因（X 轴，左->右）:", paste(GENES, collapse = " -> "), "\n")
cat("剔除供体:", paste(EXCLUDE_AGES, collapse = ", "), "\n")
cat("\n胚胎分组配色：\n"); print(cols.emb.groups[GROUP_LEVELS_EMB])
cat("\n成年分组配色：\n"); print(cols.adu.groups[GROUP_LEVELS_ADU])

In [ ]:
# ---- 2. 读取每细胞表，剔除 70/30years，计算 sharedY 上限 ----------------------
percell_all <- readRDS(PERCELL_RDS)
stopifnot(all(paste0(GENES, "_lognorm") %in% colnames(percell_all)))
excl_tab <- table(percell_all$donor_age[percell_all$donor_age %in% EXCLUDE_AGES])
cat("载入 MG 总细胞数（含 70years）:", nrow(percell_all), "\n剔除细胞数：\n"); print(excl_tab)

percell <- percell_all[!(percell_all$donor_age %in% EXCLUDE_AGES), ]
percell$period <- factor(as.character(percell$period), levels = c("Embryonic", "Adult"))
emb <- percell[percell$period == "Embryonic", ]
adu <- percell[percell$period == "Adult", ]
cat("\n剔除后 MG 细胞数：胚胎", nrow(emb), "| 成年", nrow(adu), "| 合计", nrow(percell), "\n")
cat("\n[Embryonic] 分组细胞数：\n")
print(table(droplevels(factor(as.character(emb$stage_region), levels = GROUP_LEVELS_EMB))))
cat("\n[Adult] 分组细胞数：\n")
print(table(droplevels(factor(as.character(adu$stage_region), levels = GROUP_LEVELS_ADU))))

## sharedY 上限（转置后为 X 轴/表达轴）：跨两期全局最大值，与 _5 stage 版同一基准
gene_y_max <- vapply(GENES, function(g) max(percell[[paste0(g, "_lognorm")]]), numeric(1))
cat("\n每基因跨两期全局最大 lognorm（剔除 70/30years 后）：\n"); print(round(gene_y_max, 3))

build_mini_seurat <- function(df, group_levels) {
  counts_mat <- Matrix::t(as(as.matrix(df[, paste0(GENES, "_counts")]), "dgCMatrix"))
  data_mat   <- Matrix::t(as(as.matrix(df[, paste0(GENES, "_lognorm")]), "dgCMatrix"))
  rownames(counts_mat) <- GENES; colnames(counts_mat) <- df$cell
  rownames(data_mat)   <- GENES; colnames(data_mat)   <- df$cell
  mini_meta <- data.frame(stage_region = factor(as.character(df$stage_region), levels = group_levels),
                          row.names = df$cell)
  mini <- CreateSeuratObject(counts = counts_mat, meta.data = mini_meta)
  mini[["RNA"]]$data <- data_mat
  Idents(mini) <- "stage_region"
  mini
}

In [ ]:
# ---- 3. 转置版：X 轴 = 基因，Y 轴 = 分组，小提琴横向 ---------------------------
make_rotated_violin <- function(df, group_levels, group_cols, title_str) {
  mini <- build_mini_seurat(df, group_levels)

  ## geom_blank：每基因一行，表达值 = 跨两期全局上限 + 余量（转置后撑 X 轴范围）
  blank_df <- data.frame(
    feature    = factor(GENES, levels = GENES),
    ident      = group_levels[1],
    expression = gene_y_max + pmax(0.001 * gene_y_max, 0.01)
  )

  p <- VlnPlot(mini, features = GENES, group.by = "stage_region",
               stack = TRUE, fill.by = "ident",
               cols = group_cols[group_levels],
               pt.size = 0, flip = FALSE) +
    geom_blank(data = blank_df, aes(x = expression, y = ident), inherit.aes = FALSE) +
    ## 替换默认 X（表达）标度：下限统一 0，上限由数据 + geom_blank 决定（每基因面板独立）
    scale_x_continuous(expand = c(0, 0), limits = c(0, NA), oob = squish,
                       labels = function(x) c(rep("", length(x) - 2), x[length(x) - 1], "")) +
    ## Y 轴（分组）：Early-central 在顶部，自上而下按指定顺序
    scale_y_discrete(limits = rev(group_levels)) +
    labs(title = title_str,
         subtitle = sprintf("MG cells: n = %s | young = 21/25/32y, old = 67/75/86y | per-gene x-axis shared between periods",
                            format(nrow(df), big.mark = ","))) +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
          plot.subtitle = element_text(hjust = 0.5, size = 8.5),
          strip.text.x = element_text(angle = 0, face = "bold"))

  ## 核对：基因列顺序 / Y 轴分组顺序（自上而下）/ 填充色 / 每基因 X（表达）范围
  bd <- ggplot_build(p)
  bp <- bd$layout$panel_params[[1]]
  cat(sprintf("[%s] X 轴（基因列，左->右）: %s\n", title_str,
              paste(levels(factor(p$data$feature, levels = GENES)), collapse = " | ")))
  cat(sprintf("[%s] Y 轴（分组，自上而下）: %s\n", title_str,
              paste(rev(bp$y$get_labels()), collapse = " | ")))
  ## 填充色逐 Y 槽位核对（Y 轴反转：底部 = 最后一个分组）
  fill_by_y <- vapply(seq_along(group_levels), function(i) {
    v <- as.character(bd$data[[1]]$fill[bd$data[[1]]$y == i])
    if (length(v)) unique(v)[1] else NA_character_
  }, character(1))
  stopifnot(identical(fill_by_y, unname(as.character(group_cols[rev(group_levels)]))))
  cat(sprintf("[%s] 填充色逐槽位校验通过（底部->顶部 %d 槽位）\n", title_str, length(group_levels)))
  xr <- do.call(rbind, lapply(bd$layout$panel_params, function(pp) pp$x$get_limits()))
  rownames(xr) <- GENES
  cat(sprintf("[%s] 每基因 X（表达）轴范围：\n", title_str)); print(round(xr, 4))
  invisible(list(plot = p, xranges = xr))
}

vln_emb <- make_rotated_violin(emb, GROUP_LEVELS_EMB, cols.emb.groups, "Embryonic MG")
vln_adu <- make_rotated_violin(adu, GROUP_LEVELS_ADU, cols.adu.groups, "Adult MG")

## 校验：两页每基因表达轴范围逐一相等
cat("\n两页表达轴（X）范围是否一致（逐基因）：\n")
ok <- vapply(GENES, function(g) isTRUE(all.equal(vln_emb$xranges[g, ], vln_adu$xranges[g, ])),
             logical(1))
print(ok); stopifnot(all(ok))
cat("校验通过：同一基因胚胎页与成年页表达轴范围完全一致\n")

print(vln_emb$plot)
print(vln_adu$plot)

In [ ]:
# ---- 4. 输出 PDF（页 1 胚胎 / 页 2 成年）--------------------------------------
pdf_file <- file.path(PLOT_DIR, paste0(DATE_STR, "_MG_ABCB11_TTC39B_TFRC_stacked_violin_sharedY_rotated.pdf"))
pdf(pdf_file, width = 7, height = 5.5, useDingbats = FALSE)
print(vln_emb$plot)
print(vln_adu$plot)
dev.off()
cat("已输出转置版 PDF（2 页）:", basename(pdf_file), "\n")
cat("文件大小 (MB):", round(file.size(pdf_file) / 1024^2, 3), "\n")
stopifnot(!file.exists(file.path(PLOT_DIR, "2026-10-21_5_MG_ABCB11_TTC39B_TFRC_stacked_violin_sharedY.pdf")) ||
          file.size(file.path(PLOT_DIR, "2026-10-21_5_MG_ABCB11_TTC39B_TFRC_stacked_violin_sharedY.pdf")) > 0)
cat("原 _5 stage 版文件未被改动\n")

In [ ]:
# ---- 5. 文件清单与 sessionInfo -----------------------------------------------
fi <- data.frame(file = basename(pdf_file),
                 size_MB = round(file.size(pdf_file) / 1024^2, 3))
print(fi, row.names = FALSE)
cat("\n全部完成。\n\n")
sessionInfo()